# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely."""
    import re
    # Strip trigger words and trailing punctuation
    cleaned = re.sub(r'(?i)(calculate|compute|solve|what is|eval|=\?)', '', expression).strip().rstrip('?.')
    try:
        # Only allow safe characters
        if not re.match(r'^[\d\s\+\-\*\/\%\(\)\.]+$', cleaned):
            return "Error: Invalid characters in expression"
        result = eval(cleaned)
        return str(result)
    except Exception as e:
        return f"Error in calculation: {e}"

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract meaningful keywords from text (filters stopwords)."""
    STOPWORDS = {
        "from", "this", "that", "with", "have", "will", "been", "they",
        "their", "there", "what", "when", "where", "which", "about",
        "into", "more", "also", "than", "then", "some", "such", "each",
        "extract", "keywords", "text", "sentence", "following"
    }
    try:
        words = text.split()
        keywords = list(dict.fromkeys(
            w.lower().strip(".,!?") for w in words
            if len(w) > 4 and w.lower() not in STOPWORDS
        ))
        return keywords[:5]
    except Exception:
        return []


# 🛠️ TOOL 3: Text Summarizer (Bonus)

def summarize(text: str) -> str:
    """Return the first 2 sentences as a simple extractive summary."""
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    summary = " ".join(sentences[:2])
    return summary if summary else text[:200]


# 🛠️ TOOL 4: Word Counter (Bonus)

def word_count(text: str) -> dict:
    """Count words, characters, and sentences in text."""
    import re
    words = text.split()
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return {
        "words": len(words),
        "characters": len(text),
        "sentences": len(sentences)
    }


# 🛠️ TOOL 5: Unit Converter (Bonus)

def convert_units(query: str) -> str:
    """Convert between common units (km↔miles, kg↔lbs, C↔F)."""
    import re
    q = query.lower()
    num = re.search(r'[\d.]+', q)
    if not num:
        return "Error: No number found in query"
    val = float(num.group())

    if "km" in q and "mile" in q:
        result = val * 0.621371 if "km to" in q else val / 0.621371
        unit = "miles" if "km to" in q else "km"
    elif "kg" in q and ("lb" in q or "pound" in q):
        result = val * 2.20462 if "kg to" in q else val / 2.20462
        unit = "lbs" if "kg to" in q else "kg"
    elif "celsius" in q and "fahrenheit" in q:
        result = (val * 9/5) + 32
        unit = "°F"
    elif "fahrenheit" in q and "celsius" in q:
        result = (val - 32) * 5/9
        unit = "°C"
    else:
        return "Error: Unsupported unit conversion"
    return f"{val} → {result:.4f} {unit}"

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [3]:
# 🤖 AGENT FUNCTION — with improved routing, logging, and multi-tool support

import re
import sys
import logging
from datetime import datetime

# ── Logging setup (Bonus) ──────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    stream=sys.stdout,
    force=True,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("SmartAgent")

# ── Intent patterns (Bonus: regex-based, more robust than keyword matching) ─
INTENT_PATTERNS = {
    "calculation": [
        r'\bcalculate\b', r'\bcompute\b', r'\bsolve\b',
        r'\d+\s*[\+\-\*\/\%]\s*\d+',        # e.g. "20 + 5"
        r'\bwhat is\s+[\d\s\+\-\*\/]+\??$',  # "what is 10 * 3?"
    ],
    "keywords": [
        r'\bkeywords?\b', r'\bkey\s*terms?\b',
        r'\bextract\b', r'\bimportant words\b',
    ],
    "summarize": [
        r'\bsummar(ize|ise|y)\b', r'\bbriefly\b', r'\btldr\b', r'\bin short\b',
    ],
    "word_count": [
        r'\bword\s*count\b', r'\bhow many words\b', r'\bcount\s*(words|characters)\b',
    ],
    "convert": [
        r'\bconvert\b', r'\bkm to miles?\b', r'\bmiles? to km\b',
        r'\bkg to (lbs?|pounds?)\b', r'\b(celsius|°c) to (fahrenheit|°f)\b',
        r'\b(fahrenheit|°f) to (celsius|°c)\b',
    ],
}

GENERAL_RESPONSES = {
    r'\bmachine learning\b':
        "Machine Learning is a subset of AI where models learn patterns from data to make predictions or decisions without being explicitly programmed.",
    r'\bdeep learning\b':
        "Deep Learning uses multi-layered neural networks to model complex patterns in data, enabling breakthroughs in vision, speech, and NLP.",
    r'\bneural network\b':
        "Neural networks are computing systems inspired by biological neurons, consisting of layers of interconnected nodes that learn from data.",
    r'\bpython\b':
        "Python is a high-level, interpreted programming language widely used in data science, AI, web development, and automation.",
    r'\bartificial intelligence\b|\b\bai\b':
        "Artificial Intelligence (AI) is the simulation of human intelligence processes by machines, including learning, reasoning, and problem-solving.",
}


def detect_intent(query: str) -> str:
    """Detect the intent of a query using regex pattern matching."""
    q = query.lower()
    for intent, patterns in INTENT_PATTERNS.items():
        if any(re.search(p, q) for p in patterns):
            return intent
    return "general"


def agent(query: str) -> dict:
    """
    Single-agent smart assistant with conditional routing.

    Routes to:
      - calculation  → Calculator Tool
      - keywords     → Keyword Extractor Tool
      - summarize    → Summarizer Tool  (Bonus)
      - word_count   → Word Count Tool  (Bonus)
      - convert      → Unit Converter   (Bonus)
      - general      → Direct response
    """
    logger.info(f"Query received: '{query}'")

    if not query or not query.strip():
        logger.warning("Empty query received")
        return {"type": "error", "result": "Query cannot be empty"}

    intent = detect_intent(query)
    logger.info(f"Intent detected: {intent}")

    try:
        if intent == "calculation":
            result = calculator(query)
            logger.info(f"Calculator result: {result}")
            return {"type": "calculation", "result": result, "expression": query}

        elif intent == "keywords":
            # Extract the actual text to analyze (after "from" or "in")
            match = re.search(r'(?:from|in)\s+(.+)', query, re.IGNORECASE)
            text = match.group(1) if match else query
            result = extract_keywords(text)
            logger.info(f"Keywords extracted: {result}")
            return {"type": "keywords", "result": result, "source_text": text}

        elif intent == "summarize":
            match = re.search(r'(?:summarize|summary of|summarise)\s+(.+)', query, re.IGNORECASE)
            text = match.group(1) if match else query
            result = summarize(text)
            logger.info(f"Summary generated")
            return {"type": "summarize", "result": result}

        elif intent == "word_count":
            match = re.search(r'(?:count|in)\s+["\']?(.+)["\']?$', query, re.IGNORECASE)
            text = match.group(1) if match else query
            result = word_count(text)
            logger.info(f"Word count: {result}")
            return {"type": "word_count", "result": result}

        elif intent == "convert":
            result = convert_units(query)
            logger.info(f"Conversion result: {result}")
            return {"type": "convert", "result": result}

        else:
            # General knowledge response
            q_lower = query.lower()
            for pattern, response in GENERAL_RESPONSES.items():
                if re.search(pattern, q_lower):
                    logger.info("Matched general knowledge pattern")
                    return {"type": "general", "result": response}

            logger.info("No specific pattern matched — returning default response")
            return {
                "type": "general",
                "result": f"I received your query: '{query}'. I can help with calculations, keyword extraction, summarization, word counting, and unit conversion. Try asking me to 'calculate 10 * 5' or 'extract keywords from ...'."
            }

    except Exception as e:
        logger.error(f"Agent error: {e}")
        return {"type": "error", "result": f"An error occurred: {str(e)}"}

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [4]:
# 🧪 Test Cases — Required + Bonus

import json

queries = [
    # Required
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    # Bonus — additional tools
    "compute 144 / 12",
    "what is 15 * 4?",
    "word count in The quick brown fox jumps over the lazy dog",
    "convert 10 km to miles",
    "convert 100 celsius to fahrenheit",
    "What is deep learning?",
    "",                         # edge case: empty query
]

print("=" * 60)
for q in queries:
    print(f"\nQuery   : {q!r}")
    response = agent(q)
    print(f"Response: {json.dumps(response, indent=2)}")
    print("-" * 60)


Query   : 'Calculate 20 + 5'
03:46:05 [INFO] Query received: 'Calculate 20 + 5'
03:46:05 [INFO] Intent detected: calculation
03:46:05 [INFO] Calculator result: 25
Response: {
  "type": "calculation",
  "result": "25",
  "expression": "Calculate 20 + 5"
}
------------------------------------------------------------

Query   : 'Extract keywords from Artificial Intelligence is transforming industries'
03:46:05 [INFO] Query received: 'Extract keywords from Artificial Intelligence is transforming industries'
03:46:05 [INFO] Intent detected: keywords
03:46:05 [INFO] Keywords extracted: ['artificial', 'intelligence', 'transforming', 'industries']
Response: {
  "type": "keywords",
  "result": [
    "artificial",
    "intelligence",
    "transforming",
    "industries"
  ],
  "source_text": "Artificial Intelligence is transforming industries"
}
------------------------------------------------------------

Query   : 'What is machine learning?'
03:46:05 [INFO] Query received: 'What is machine le

In [5]:
# 🎯 Interactive Mode
# Run this cell manually in Jupyter to try the agent interactively

# while True:
#     user_input = input("Enter query (type 'exit' to stop): ")
#     if user_input.lower() == "exit":
#         break
#     print("Response:", agent(user_input))

# Quick demo instead (uncomment the loop above when running interactively):
demo_queries = [
    "calculate 99 * 3",
    "extract keywords from Natural Language Processing enables computers to understand human text",
    "convert 5 kg to lbs",
]
for q in demo_queries:
    print(f">>> {q}")
    print(agent(q))
    print()

>>> calculate 99 * 3
03:46:05 [INFO] Query received: 'calculate 99 * 3'
03:46:05 [INFO] Intent detected: calculation
03:46:05 [INFO] Calculator result: 297
{'type': 'calculation', 'result': '297', 'expression': 'calculate 99 * 3'}

>>> extract keywords from Natural Language Processing enables computers to understand human text
03:46:05 [INFO] Query received: 'extract keywords from Natural Language Processing enables computers to understand human text'
03:46:05 [INFO] Intent detected: keywords
03:46:05 [INFO] Keywords extracted: ['natural', 'language', 'processing', 'enables', 'computers']
{'type': 'keywords', 'result': ['natural', 'language', 'processing', 'enables', 'computers'], 'source_text': 'Natural Language Processing enables computers to understand human text'}

>>> convert 5 kg to lbs
03:46:05 [INFO] Query received: 'convert 5 kg to lbs'
03:46:05 [INFO] Intent detected: convert
03:46:05 [INFO] Conversion result: 5.0 → 11.0231 lbs
{'type': 'convert', 'result': '5.0 → 11.0231 lbs